In [2]:
%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection'

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    train_dataset_path: Path
    val_dataset_path: Path
    params_image_size: list
    params_batch_size: int
    params_validation_split: float

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        params = self.params

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            train_dataset_path=Path(config.train_dataset_path),
            val_dataset_path=Path(config.val_dataset_path),
            params_image_size=params.IMAGE_SIZE,
            params_batch_size=params.BATCH_SIZE,
            params_validation_split=params.VALIDATION_SPLIT
        )

        return data_transformation_config

In [12]:
import os
import tensorflow as tf
from pathlib import Path
from cnnClassifier import logger


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def transform_and_save_data(self):

        logger.info("Loading train and test datasets...")

        img_size = tuple(self.config.params_image_size[:-1])

        train_path = os.path.join(self.config.data_path, "train")
        test_path = os.path.join(self.config.data_path, "test")

        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_path,
            image_size=img_size,
            batch_size=self.config.params_batch_size,
            shuffle=True,
            label_mode="categorical"
        )

        test_ds = tf.keras.utils.image_dataset_from_directory(
            test_path,
            image_size=img_size,
            batch_size=self.config.params_batch_size,
            shuffle=False,
            label_mode="categorical"
        )

        logger.info(f"Classes: {train_ds.class_names}")

        # ----------------------------------
        # Data Augmentation (Training Only)
        # ----------------------------------

        data_augmentation = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.03)      # ≈10°
        ])

        # Normalize [0,255] -> [-1,1]
        normalization = tf.keras.layers.Rescaling(
            scale=1.0 / 127.5,
            offset=-1
        )

        AUTOTUNE = tf.data.AUTOTUNE

        train_ds = train_ds.map(
            lambda x, y: (
                normalization(
                    data_augmentation(x, training=True)
                ),
                y
            ),
            num_parallel_calls=AUTOTUNE
        )

        test_ds = test_ds.map(
            lambda x, y: (
                normalization(x),
                y
            ),
            num_parallel_calls=AUTOTUNE
        )

        train_ds = train_ds.prefetch(AUTOTUNE)
        test_ds = test_ds.prefetch(AUTOTUNE)

        logger.info("Saving transformed training dataset...")
        tf.data.Dataset.save(
            train_ds,
            str(self.config.train_dataset_path)
        )

        logger.info("Saving transformed testing dataset...")
        tf.data.Dataset.save(
            test_ds,
            str(self.config.val_dataset_path)
        )

        logger.info("Data transformation completed successfully.")

In [13]:
STAGE_NAME = "Data Transformation Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.transform_and_save_data()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-07-11 15:27:47,841: INFO: 3634220854: >>>>>> stage Data Transformation Stage started <<<<<<]
[2026-07-11 15:27:47,847: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 15:27:47,851: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-11 15:27:47,854: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-07-11 15:27:47,854: INFO: common: created directory at: artifacts]
[2026-07-11 15:27:47,856: INFO: common: created directory at: artifacts/data_transformation]
[2026-07-11 15:27:47,856: INFO: 1318346227: Loading train and test datasets...]
Found 13898 files belonging to 22 classes.
Found 1546 files belonging to 22 classes.
[2026-07-11 15:27:48,525: INFO: 1318346227: Classes: ['Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous', 'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites', 'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea', 'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage', 'Tinea', 'Unknown_Normal', 